# 1. See your data and simulate a frame
**Before you start:** select the **Python (DECODE Intel)** kernel. Run each cell from top to bottom (Shift+Enter). If you change `config.yaml`, rerun the first cell to reload it. Nothing is saved unless a cell tells you it will save.
**Goal:** understand what goes into a DECODE simulation. First look at your two channels; then generate an example frame. This notebook does not train a model or change a TIFF.


## A. Look at a real raw frame
The frame is split down the middle. Both halves use the same brightness scale. The number of image pixels (height × width) is different from the physical size of each pixel (nm/pixel).

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import yaml

# Start Jupyter from the project folder or decode_walkthrough/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'decode_walkthrough' / 'config.yaml').is_file():
    ROOT = ROOT.parent
config_path = ROOT / 'decode_walkthrough' / 'config.yaml'
if not config_path.is_file():
    raise FileNotFoundError('Start Jupyter in the project folder or decode_walkthrough/.')
with config_path.open() as f:
    CFG = yaml.safe_load(f)
def project_path(value):
    if value is None: return None
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p
EXAMPLE = project_path(CFG['paths']['example_folder'])
WORK = project_path(CFG['paths']['work_folder'])
print('Project:', ROOT, '| Config:', config_path, '| Example exists:', EXAMPLE.exists())


Project: /Users/rohan/Documents/projects/mphys | Config: /Users/rohan/Documents/projects/mphys/decode_walkthrough/config.yaml | Example exists: True


In [ ]:
# Choose a TIFF to inspect; the default is the first available low-density TIFF.
# To choose your own, set own_data.raw_tiff in config.yaml.
movie_path = project_path(CFG['own_data']['raw_tiff'])
if movie_path is None:
    candidates = sorted((ROOT / 'low_density_examples').glob('*.tif'))
    if not candidates: raise FileNotFoundError('Set own_data.raw_tiff in config.yaml')
    movie_path = candidates[0]
frame_index = CFG['comparison']['frame_index']
with tifffile.TiffFile(movie_path) as tif:
    if frame_index >= len(tif.pages): raise ValueError('frame_index exceeds number of TIFF pages')
    raw = tif.pages[frame_index].asarray()
    print('File:', movie_path.name, '| frames:', len(tif.pages), '| shape:', raw.shape, '| dtype:', raw.dtype)
if raw.ndim != 2 or raw.shape[1] % 2: raise ValueError('Expected grayscale frame with equal-width left/right channels.')
left, right = np.split(raw, 2, axis=1)
print('Each channel:', left.shape, 'pixels (height, width).')
limits = CFG['comparison']
vmin, vmax = limits['display_min_adu'], limits['display_max_adu']
if vmin is None and vmax is None: vmin, vmax = np.percentile(raw, limits['percentiles'])
elif vmin is None or vmax is None or vmin >= vmax: raise ValueError('Set both display limits, or leave both null.')
fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, image, label in zip(axes,(left,right),('Left channel','Right channel')):
    im=ax.imshow(image,cmap='gray',vmin=vmin,vmax=vmax); ax.set_title(label)
    print(label,'ADU percentiles 1,50,99:',np.percentile(image,[1,50,99]).round(1))
fig.colorbar(im,ax=axes,label='camera counts (ADU)'); plt.show()


**Try it:** choose another file by setting `own_data.raw_tiff` in `config.yaml`, relative to the project folder, and rerun from the top. Auto contrast can make separate recordings look different; set both `comparison.display_min_adu` and `display_max_adu` to fixed values to compare files fairly.

## B. Generate simulated example frames
DECODE combines emitter positions, a spline PSF, background and camera noise. We use the **bundled example calibration and EMCCD camera**, not your microscope. Simulation output below is in DECODE’s photon units, while your raw frame above is camera counts (ADU); do not compare the numbers directly.


In [ ]:
import torch
import decode
import decode.utils.param_io
from decode.neuralfitter.train.random_simulation import setup_random_simulation
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())


In [ ]:
param = decode.utils.param_io.load_params(str(project_path(CFG['paths']['example_parameters'])))
param.InOut.calibration_file = str(project_path(CFG['paths']['example_calibration']))
sim_cfg = CFG['example_simulation']
param.Hardware.device_simulation = sim_cfg['device']
param.Hardware.device = sim_cfg['device']
param.Simulation.emitter_av = sim_cfg['emitter_average']
param.Simulation.bg_uniform = sim_cfg['background_photons']
param.Simulation.intensity_mu_sig = sim_cfg['brightness_mean_std']
param.HyperParameter.pseudo_ds_size = sim_cfg['last_frame_index']  # inclusive
param.TestSet.test_size = sim_cfg['last_frame_index']
torch.manual_seed(sim_cfg['seed'])
np.random.seed(sim_cfg['seed'])  # DECODE uses both NumPy and PyTorch random generators
print('Example PSF:', param.InOut.calibration_file)
print('Emitter average:',param.Simulation.emitter_av,'| background:',param.Simulation.bg_uniform)


In [ ]:
sim, _ = setup_random_simulation(param)
emitters, frames, backgrounds = sim.sample()
frames = frames.detach().cpu().numpy()
print('Simulated frames:', frames.shape, '| convert to photons:', param.Camera.convert2photons)
plt.figure(figsize=(6,5)); plt.imshow(frames[0], cmap='gray'); plt.colorbar(label='model output'); plt.title('DECODE example simulation'); plt.show()


**Try it:** change `example_simulation.emitter_average` or `background_photons` in `config.yaml`, then rerun from the top. `emitter_average` is the number per **40 × 40 pixel simulation**, not your full camera frame. More background can make spots harder to distinguish.

**Next:** notebook 2 explains the PSF and what the lab needs to provide.
